In [1]:

import numpy as np
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

# 🔹 Parameters
input_dim = 20
latent_dim = 2

# 🔹 Encoder
inputs = Input(shape=(input_dim,))
encoded = Dense(10, activation='relu')(inputs)
latent = Dense(latent_dim)(encoded)

# 🔹 Decoder
decoded = Dense(10, activation='relu')(latent)
outputs = Dense(input_dim, activation='sigmoid')(decoded)

# 🔹 VAE Model
vae = Model(inputs, outputs)
vae.compile(optimizer='adam', loss='mse')

# 🔹 Dummy Data
X_train = np.random.rand(1000, input_dim)

# 🔹 Train
vae.fit(X_train, X_train, epochs=10, batch_size=32)

# 🔹 Test Reconstruction
sample = np.random.rand(1, input_dim)
reconstructed = vae.predict(sample)

print("Original:", sample)
print("Reconstructed:", reconstructed)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0838
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0826
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0818
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0811
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0803
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0795
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0787
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0780
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0775
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0770
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
Original: [[2.17284396e-01 4.62454716e-01 5.42996589e-04 7.64284851e-02
  6.81542298e-02 9.96343052e-03 3.80888314e-01 8.35927259e-01
  9.54035452e-01 1.45253606e-01 2.81545727e-02 9.71059145e-01
  2.16953095e-01 2.07189285e-01 6.56237535e-01 4.67944023e-01
  3.66861190e-03 9.09358784e-01 9.30175111e-01 8.7957

In [2]:

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# 🔹 1. Generator Model
generator = Sequential([
    Dense(16, activation='relu', input_shape=(10,)),
    Dense(1, activation='sigmoid')
])

# 🔹 2. Discriminator Model
discriminator = Sequential([
    Dense(16, activation='relu', input_shape=(1,)),
    Dense(1, activation='sigmoid')
])

discriminator.compile(optimizer=Adam(learning_rate=0.001),
                      loss='binary_crossentropy',
                      metrics=['accuracy'])

# 🔹 3. Combined GAN Model
discriminator.trainable = False
gan = Sequential([generator, discriminator])

gan.compile(optimizer=Adam(learning_rate=0.001),
            loss='binary_crossentropy')

# 🔹 4. Training Loop
epochs = 200
batch_size = 32

for epoch in range(epochs):

    # -------- Train Discriminator --------
    real_data = np.random.rand(batch_size, 1)

    noise = np.random.rand(batch_size, 10)
    fake_data = generator(noise, training=False)   # faster than predict

    X = np.vstack((real_data, fake_data))
    y = np.vstack((np.ones((batch_size, 1)),
                   np.zeros((batch_size, 1))))

    discriminator.trainable = True
    d_loss, d_acc = discriminator.train_on_batch(X, y)

    # -------- Train Generator --------
    noise = np.random.rand(batch_size, 10)
    misleading_labels = np.ones((batch_size, 1))  # trick discriminator

    discriminator.trainable = False
    g_loss = gan.train_on_batch(noise, misleading_labels)

    # -------- Print Progress --------
    if epoch % 20 == 0:
        print(f"Epoch {epoch} | D Loss: {d_loss:.4f}, Acc: {d_acc:.4f} | G Loss: {g_loss:.4f}")

# 🔹 5. Generate New Data
noise = np.random.rand(1, 10)
generated_data = generator(noise, training=False)

print("\nGenerated Data:", generated_data)

c:\Users\palak\anaconda3\envs\dl_env\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 0 | D Loss: 0.6836, Acc: 0.5000 | G Loss: 0.7462
Epoch 20 | D Loss: 0.6863, Acc: 0.5737 | G Loss: 0.7373
Epoch 40 | D Loss: 0.6877, Acc: 0.6132 | G Loss: 0.7297
Epoch 60 | D Loss: 0.6897, Acc: 0.6309 | G Loss: 0.7212
Epoch 80 | D Loss: 0.6915, Acc: 0.5557 | G Loss: 0.7130
Epoch 100 | D Loss: 0.6934, Acc: 0.4981 | G Loss: 0.7057
Epoch 120 | D Loss: 0.6943, Acc: 0.4742 | G Loss: 0.7001
Epoch 140 | D Loss: 0.6944, Acc: 0.4772 | G Loss: 0.6969
Epoch 160 | D Loss: 0.6937, Acc: 0.4780 | G Loss: 0.6956
Epoch 180 | D Loss: 0.6926, Acc: 0.5018 | G Loss: 0.6958

Generated Data: tf.Tensor([[0.37225354]], shape=(1, 1), dtype=float32)


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

# 🔹 GCN Model
class SimpleGCN(nn.Module):
    def __init__(self):
        super(SimpleGCN, self).__init__()
        self.fc = nn.Linear(10, 2)

    def forward(self, x, adj):
        x = torch.matmul(adj, x)  # aggregate neighbors
        x = self.fc(x)
        return x

# 🔹 Dummy Graph Data
num_nodes = 5
features = torch.rand(num_nodes, 10)

# Adjacency matrix (example)
adj = torch.tensor([
    [1,1,0,0,0],
    [1,1,1,0,0],
    [0,1,1,1,0],
    [0,0,1,1,1],
    [0,0,0,1,1]
], dtype=torch.float32)

labels = torch.tensor([0,1,0,1,0])

# 🔹 Model + Loss
model = SimpleGCN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 🔹 Training
for epoch in range(100):
    optimizer.zero_grad()
    output = model(features, adj)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()

print("Output:\n", output)

Output:
 tensor([[1.1366, 0.2930],
        [0.8747, 0.8533],
        [1.1966, 0.6647],
        [0.5432, 1.0497],
        [0.8051, 0.4894]], grad_fn=<AddmmBackward0>)
